# Dataset Size Comparison — Presentation Visual

Goal: show a reader, at a glance, that our public ADME dataset (3,521 compounds) is smaller than the confidential, industry-scale dataset used in the comparison paper (25,000+ compounds — at least ~7.1x larger).

Three mockups below. Option 3 (grouped bar chart) is the current pick.

1. Dumbbell / lollipop
2. Pictogram / icon array
3. Grouped bar chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Real numbers: paper's dataset is confidential/industry-scale, known only as a lower bound.
# The size gap ranges 2x-7.1x depending on which split/comparison point is used.
OURS_N = 3521
PAPER_N_MIN = 25000
RATIO_MIN = PAPER_N_MIN / OURS_N
RATIO_LOW = 2.0

PAPER_LABEL = "Comparison data\n(confidential, industry-scale)"
OURS_LABEL = "Our dataset\n(public)"

COLOR_PAPER = "#94A3B8"
COLOR_OURS = "#2563EB"

print(f"Ratio range: {RATIO_LOW:.1f}x - {RATIO_MIN:.1f}x (dependent on time)")

## Option 1: Dumbbell / lollipop

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))

y = 0
ax.plot([OURS_N, PAPER_N_MIN], [y, y], color="#CBD5E1", linewidth=3, zorder=1)
ax.scatter([OURS_N], [y], color=COLOR_OURS, s=400, zorder=2, label=OURS_LABEL)
ax.scatter([PAPER_N_MIN], [y], color=COLOR_PAPER, s=400, zorder=2, label=PAPER_LABEL)

ax.annotate(f"{OURS_N:,}", (OURS_N, y), textcoords="offset points", xytext=(0, 22), ha="center", fontsize=12, fontweight="bold", color=COLOR_OURS)
ax.annotate(f"{PAPER_N_MIN:,}+", (PAPER_N_MIN, y), textcoords="offset points", xytext=(0, 22), ha="center", fontsize=12, fontweight="bold", color="#475569")

mid_x = (OURS_N + PAPER_N_MIN) / 2
ax.annotate(f"{RATIO_MIN:.1f}×+ smaller", (mid_x, y), textcoords="offset points", xytext=(0, -30), ha="center", fontsize=13, fontweight="bold", color="#1E293B")

ax.set_xlim(0, PAPER_N_MIN * 1.15)
ax.set_ylim(-1, 1)
ax.set_yticks([])
ax.set_xlabel("Compounds (N)")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.45), ncol=2, frameon=False)
ax.set_title("Dataset size: ours vs. the paper", fontsize=13, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "dataset_size_dumbbell.png", dpi=200, bbox_inches="tight")
plt.show()

## Option 2: Pictogram / icon array

1 square = 100 compounds. No axis literacy required — the reader just counts rows.

In [ ]:
UNIT = 500
COLS = 10

def n_squares(n, unit):
    return int(np.ceil(n / unit))

def draw_pictogram(ax, n, unit, cols, color, label, y_offset, plus=False):
    count = n_squares(n, unit)
    for i in range(count):
        row = i // cols
        col = i % cols
        ax.add_patch(mpatches.Rectangle((col, -row - y_offset), 0.8, 0.8, color=color))
    rows_used = int(np.ceil(count / cols))
    suffix = "+" if plus else ""
    ax.text(cols / 2, -rows_used - y_offset - 0.6, f"{label}\n{n:,}{suffix} compounds", ha="center", va="top", fontsize=11, fontweight="bold")
    return rows_used

fig, ax = plt.subplots(figsize=(9, 6))

rows_ours = draw_pictogram(ax, OURS_N, UNIT, COLS, COLOR_OURS, OURS_LABEL, y_offset=0)
gap = rows_ours + 2.2
rows_paper = draw_pictogram(ax, PAPER_N_MIN, UNIT, COLS, COLOR_PAPER, PAPER_LABEL, y_offset=gap, plus=True)

ax.set_xlim(-0.5, COLS + 0.5)
ax.set_ylim(-(gap + rows_paper + 2), 1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"1 square = {UNIT} compounds  —  at least {RATIO_MIN:.1f}× smaller", fontsize=13, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "dataset_size_pictogram.png", dpi=200, bbox_inches="tight")
plt.show()

## Option 3: Grouped bar chart

Three variants of how to encode the 2x-7.1x range next to the paper's bar.

### 3a: double-headed arrow (current)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar([OURS_LABEL, PAPER_LABEL], [OURS_N, PAPER_N_MIN], color=[COLOR_OURS, COLOR_PAPER], width=0.5)

ax.annotate(f"{OURS_N:,}", (bars[0].get_x() + bars[0].get_width() / 2, OURS_N), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=12, fontweight="bold")
ax.annotate(f"{PAPER_N_MIN:,}+", (bars[1].get_x() + bars[1].get_width() / 2, PAPER_N_MIN), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=12, fontweight="bold")

# arrow spans the ratio range (2x-7.1x of our N), not down to our N itself — the two datasets are never the same size
ratio_low_n = RATIO_LOW * OURS_N
ax.annotate("", xy=(1, PAPER_N_MIN), xytext=(1, ratio_low_n), arrowprops=dict(arrowstyle="<->", color="#1E293B"))
ax.text(1.32, (ratio_low_n + PAPER_N_MIN) / 2, f"{RATIO_LOW:.0f}–{RATIO_MIN:.1f}×\nsmaller\n(dependent on time)", ha="left", va="center", fontsize=12, fontweight="bold")

ax.set_ylabel("Compounds (N)")
ax.set_ylim(0, PAPER_N_MIN * 1.2)
ax.set_xlim(-0.6, 2.1)
ax.tick_params(axis="x", labelsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Dataset size: Public data vs. Confidential data", fontsize=13, fontweight="bold")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "dataset_size_barchart.png", dpi=200, bbox_inches="tight")
plt.show()